# 3. Knowledge Base Construction (RAG Pipeline)

This notebook builds the local vector database used by the MindCare agent
for Retrieval-Augmented Generation (RAG). Instead of relying solely on
the LLM's internal knowledge, we ground responses in verified clinical
sources to prevent hallucinations.

### Objectives
1. Ingest heterogeneous files (PDF, TXT) from the source directory.
2. Chunk text into manageable pieces (800 chars with 100 overlap).
3. Generate embeddings using Mistral's embedding model.
4. Index vectors in a local FAISS database for millisecond-scale retrieval.

In [14]:
import os
import time
from dotenv import load_dotenv

# --- LangChain Imports ---
from langchain_community.document_loaders import TextLoader, PyPDFLoader, CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_mistralai import MistralAIEmbeddings

# --- Configuration ---
SOURCE_DIRECTORY = "../source"       
VECTORSTORE_PATH = "../vectorstore_psychology"

# --- Load Environment Variables ---
# We explicitly tell it to look in the parent directory ("../.env")
# because this notebook is inside the 'notebooks' folder.
success = load_dotenv("../.env")

if not success:
    # Fallback: try loading from current directory just in case
    load_dotenv()

api_key = os.getenv("MISTRAL_API_KEY")

if not api_key:
    print("[ERROR] API Key missing. Make sure '.env' is in the project root.")
else:
    print("[INFO] API Key loaded successfully.")

[INFO] API Key loaded successfully.


### 1. Document Loading (Multi-format)
The system iterates through the `source` directory to load all available knowledge assets. The script automatically detects the file extension (.pdf, .txt, .csv) and selects the appropriate LangChain Loader to extract the raw text.

In [15]:
documents = []
files_found = 0

if not os.path.exists(SOURCE_DIRECTORY):
    print(f"[ERROR] Directory '{SOURCE_DIRECTORY}' does not exist.")
else:
    print(f"[INFO] Scanning directory: {SOURCE_DIRECTORY}...\n")
    
    for filename in os.listdir(SOURCE_DIRECTORY):
        file_path = os.path.join(SOURCE_DIRECTORY, filename)
        loader = None
        
        try:
            # Intelligent Loader Selection
            if filename.endswith(".txt"):
                print(f"   [LOADING] TXT: {filename}")
                loader = TextLoader(file_path, encoding='utf-8')
            elif filename.endswith(".pdf"):
                print(f"   [LOADING] PDF: {filename}")
                loader = PyPDFLoader(file_path)
            elif filename.endswith(".csv"):
                print(f"   [LOADING] CSV: {filename}")
                loader = CSVLoader(file_path, encoding='utf-8')
            
            # Load content
            if loader:
                docs = loader.load()
                documents.extend(docs)
                files_found += 1
                
        except Exception as e:
            print(f"   [ERROR] Failed to load {filename}: {e}")

    print(f"\n[SUMMARY] Total: {len(documents)} raw documents loaded from {files_found} files.")

[INFO] Scanning directory: ../source...

   [LOADING] PDF: 101_Coping_Skills_ADA.pdf


Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 17 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)


   [LOADING] PDF: JHEAP-Grounding-Techniques-to-Help-Control-Anxietypdf.pdf
   [LOADING] PDF: NCP-HL-Stress-Workbook-MHC-digital-v03-508.pdf


Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 17 0 (offset 0)
Ignoring wrong pointing object 45 0 (offset 0)


   [LOADING] TXT: psychology_guide.txt
   [LOADING] PDF: Relaxation-Skills-for-Anxiety.pdf

[SUMMARY] Total: 41 raw documents loaded from 5 files.


### 2. Text Splitting (Chunking)
LLMs have a limited context window. We use a `RecursiveCharacterTextSplitter` to break documents into "bricks" of **800 characters**.

* **Chunk Size (800):** Chosen to provide enough context for the model to understand the semantic meaning without overloading the token limit.
* **Overlap (100):** Ensures semantic continuity between chunks so context isn't lost at the cut point.

In [16]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,      # Target size for each chunk
    chunk_overlap=100,   # Overlap to maintain context
    separators=["\n\n", "\n", ".", " ", ""] # Priority for splitting
)

chunks = text_splitter.split_documents(documents)

print(f"[INFO] Splitting complete.")
print(f"   - Source files: {files_found}")
print(f"   - Total chunks created: {len(chunks)}")
print(f"   - Sample of Chunk #5: \n\n---\n{chunks[5].page_content}\n---")

[INFO] Splitting complete.
   - Source files: 5
   - Total chunks created: 142
   - Sample of Chunk #5: 

---
60. Spend time with someone 85. Chew gum positive 
86. Go people watching 61. Sit in a hot tub, sauna or pool 
87. Go to a museum 62. Read inspirational quotes 
88. Garden 63. Read self-help articles or books 
89. Think of something you accomplished 6.4. Name 3+ of your positive 
attributes 90. Focus on being in the present moment 
65. Take care of your physical 91. Write a blog 
appearance 92. Make a list of your personal coping 
66. Take responsibility for your skills 
part of a problem 93. Play a sport 
67. Turn a difficult situation into a 9.4. Volunteer 
learning experience, focus on 95. Catch yourself when you are the learning opportunity over-thinking 
68. Visit a pet store, animal shelter 96. Write a thank you card to someone or feed animals
---


## 3. Embedding & Indexing

Each chunk is sent to the Mistral embedding API, which returns a
high-dimensional vector representing its semantic meaning. FAISS
indexes these vectors for efficient similarity search at query time.

In [17]:
print("[INFO] Generating Embeddings (Mistral) and Indexing (FAISS)...")
start_time = time.time()

try:
    # 1. Initialize Embedding Model
    embeddings = MistralAIEmbeddings(api_key=api_key, model="mistral-embed")
    
    # 2. Create Vector Database (This includes the API call for embeddings)
    vector_db = FAISS.from_documents(chunks, embeddings)
    
    # 3. Save to local disk for persistence
    vector_db.save_local(VECTORSTORE_PATH)
    
    end_time = time.time()
    print(f"[SUCCESS] Vector store saved at '{VECTORSTORE_PATH}'")
    print(f"[METRICS] Processing time: {round(end_time - start_time, 2)} seconds.")
    
except Exception as e:
    print(f"[CRITICAL ERROR] Embedding failed: {e}")

[INFO] Generating Embeddings (Mistral) and Indexing (FAISS)...


c:\Users\ayoub\Desktop\ECAM\Bloc 5\3. IA\1. Projet Mindcare\MindCare_Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[SUCCESS] Vector store saved at '../vectorstore_psychology'
[METRICS] Processing time: 3.8 seconds.


## 4. Sanity Check

We query the database to verify retrieval works correctly.
This simulates the query mechanism used by ECO mode in the app.

In [18]:
# Reload DB to ensure persistence works
loaded_db = FAISS.load_local(VECTORSTORE_PATH, embeddings, allow_dangerous_deserialization=True)

test_query = "How to manage anxiety?"
print(f"[TEST] Query: '{test_query}'\n")

# Retrieve top 2 most relevant chunks based on Euclidean distance
results = loaded_db.similarity_search(test_query, k=2)

for i, doc in enumerate(results):
    print(f"--- Result {i+1} (Source: {doc.metadata['source']}) ---")
    print(doc.page_content)
    print("\n")

[TEST] Query: 'How to manage anxiety?'

--- Result 1 (Source: ../source\Relaxation-Skills-for-Anxiety.pdf) ---
pace in activity, whether walking,  
working, or even planning leisure 
activities, communicates a sense of 
urgency to the brain, raising blood 
pressure and tension in the body. 
This has an impact on our anxiety 
from day-to-day. Practice   
“slowing down” your pace of 
life consciously to reduce 
this sense of 
urgency.
Goal Setting
Set realistic goals in line with your 
life aims. Strive for balance of   
meaningful work, interpersonal (family 
and friends), and enjoyment-oriented  
goals. Remember to take one  small 
step at a time to reach larger 
goals .
Treat Mental Illness
Learn to manage anxiety using   
CBT skills. Treat other forms of 
mental illness if they interfere with your    
life. If the therapy you try does not seem 
to be working, try another therapy style


--- Result 2 (Source: ../source\Relaxation-Skills-for-Anxiety.pdf) ---
life. If the therapy you tr